In [7]:
import sqlite3
import PySimpleGUI as sg

con = sqlite3.connect('son.db')
cur = con.cursor()

login_user_name = -1
login_user_type = -1
login_id = -1


def window_main():
    layout = [[sg.Button('Create new player')],
              [sg.Button('Login page')]]
    return sg.Window('Main Window', layout)

def window_create_player():
    layout = [[sg.Text('Username', size=(10,1)), sg.Input(size=(10,1), key='name')],
              [sg.Text('Password', size=(10,1)), sg.Input(size=(10,1), key='cpassword')],
              [sg.Text('Gamertag', size=(10,1)), sg.Input(size=(10,1), key='gamertag')],
              [sg.Button('Add Player')],
              [sg.Button('Return to Create')]]
    return sg.Window('Create Player Window', layout)
             

def window_login():
    layout = [[sg.Text('Welcome to the video game store')],
              [sg.Text('Username:', size=(10,1)), sg.Input(size=(10,1), key='username')],
              [sg.Text('Password:', size=(10,1)), sg.Input(size=(10,1), key='password')], 
              [sg.Button('Login')],
              [sg.Button('Return to Create')]]
              
    return sg.Window('Login Window', layout)

def window_admin():
    layout = [[sg.Text('Welcome ' + login_user_name)], 
              [sg.Button('Games Waiting Approval')],
              [sg.Button('Logout')]]
    return sg.Window('Player Window', layout)

def window_approval():
    genres_a = ['Action', 'RPG', 'CityBuild', 'RogueLike', 'Simulation', 'Strategy']
    platforms_a = ['PC', 'PS', 'Xbox', 'Switch']
    below_prices_appr = [6, 11, 16, 21, 26, 40, 50, 100]
    below_size_appr = [5, 10, 15, 20, 25, 40, 50, 100]
    approve_waiting_games = []
    for row in cur.execute("select GameID, GameName, Genre, Price, Platforms, DownloadSize, PublishedDate from Games WHERE Games.ApprovalStatus=0 ORDER BY PublishedDate DESC"):
        approve_waiting_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
        
    
    layout = [[sg.Text('Games Waiting Approval' , size=(20,1))],
              [sg.Listbox(approve_waiting_games, size=(50,15), key='waiting_games')],
              [sg.Text('Select Genre', size=(10,1)), sg.Combo(genres_a, key='genre_selection_admin')],
              [sg.Text('Platforms', size=(10,1)), sg.Combo(platforms_a, key='platform_selection_admin')],
              [sg.Text('Below Price', size=(10,1)), sg.Combo(below_prices_appr, key='below_price_appr')],
              [sg.Text('Below Size', size=(10,1)), sg.Combo(below_size_appr, key='below_size_appr')],
              [sg.Text('Before Date', size=(15,1)), sg.Input(key='before_date', size=(15,1)), sg.CalendarButton('Choose Date', format='%Y-%m-%d')],
              [sg.Button('List')],
              [sg.Button('Approve'), sg.Button('Revoke')],
              [sg.Button('Return to Main')]]
    return sg.Window('Approve Window', layout)
def window_player():
    layout = [[sg.Text('Welcome ' + login_user_name)], 
              [sg.Button('My Account')], 
              [sg.Button('View My Wallet')], 
              [sg.Button('Library')],
              [sg.Button('Friends')],
              [sg.Button('Games')],
              [sg.Button('Gifted Games')],
              [sg.Button('Logout')]]
    return sg.Window('Player Window', layout)

def window_developer():
    layout = [[sg.Text('Welcome ' + login_user_name)],
              [sg.Button('View Balance', size=(10,1))],
              [sg.Button('Created Games')],
              [sg.Button('Create Game')],
              [sg.Button('Logout')]]
    return sg.Window('Developer Window', layout)

def window_created_games():
    genres = ['Action', 'RPG', 'CityBuild', 'RogueLike', 'Simulation', 'Strategy']
    platforms = ['PC', 'PS', 'Xbox', 'Switch']
    status_approval = [1, 0]
    below_prices_de = [6, 11, 16, 21, 26, 40, 50, 100]
    listed_games_dev = []
    # created_games = []
    # for row in cur.execute('SELECT GameName, Price, Genre, PublishedDate, Platforms, DownloadSize, Sold FROM Games, Develop WHERE Develop.GameID=Games.GameID AND DeveloperID=? ORDER BY PublishedDate', (login_id,)):
    #     created_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
    for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus FROM Games, Develop, Developers WHERE Developers.DeveloperID=Develop.DeveloperID AND Develop.GameID=Games.GameID AND Developers.DeveloperID=? ORDER BY PublishedDate DESC', (login_id,)):
            listed_games_dev.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8]))
    layout = [[sg.Text('Published Games', size=(15,1)), sg.Listbox(listed_games_dev, size=(50, 6), key='created_game')],
              [sg.Text('Select Genre', size=(10,1)), sg.Combo(genres, key='genre_selection_dev')],
              [sg.Text('Below Price', size=(10,1)), sg.Combo(below_prices_de, key='below_price_d')],
              [sg.Text('Platforms', size=(10,1)), sg.Combo(platforms, key='platform_selection_dev')],
              [sg.Text('Status of Approval', size=(10,1)), sg.Combo(status_approval, key='status_of_approve')],
              [sg.Button('Filter')],
              [sg.Button('Return to Main')]]
    return sg.Window('Created Games Window', layout)

def window_view_balance():
    cur.execute('SELECT DeveloperWallet FROM Developers WHERE DeveloperID = ?', (login_id,))
    row = cur.fetchone()
    login_balance = row[0]
    layout = [[sg.Text('Current money:', size=(20,1)), sg.Text(str(login_balance), size=(20,1),key='login_balance')],
              [sg.Text('Add money to your balance:', size=(25,1))],
              [sg.Input(size=(15,1), key='d_amount')],
              [sg.Button('Add to Balance')],
              [sg.Button('Return to Main')]]
    return sg.Window('Wallet Window', layout)
    
                
def window_my_account():
    cur.execute('SELECT Biography FROM Players WHERE PlayerID = ?', (login_id,))
    row = cur.fetchone()
    login_biography = row[0]
    layout = [[sg.Text('My biography'), sg.Text(login_biography, size=(20,1), key='login_biography')], 
              [sg.Input(key='biography_text'), sg.Button('Add Biography')],
              [sg.Button('Return to Main')]]
    return sg.Window('Account Window', layout)

def window_view_wallet():
    cur.execute('SELECT PlayerWallet FROM Players WHERE PlayerID = ?', (login_id,))
    row = cur.fetchone()
    login_credit = row[0]
    layout = [[sg.Text('Current money:', size=(30,1)), sg.Text(str(login_credit), size=(10,1),key='login_credit')],
              [sg.Text('Add money to your balance:', size=(10,1))],
              [sg.Input(size=(10,1), key='p_amount')],
              [sg.Button('Add')],
              [sg.Button('Return to Main')]]
    return sg.Window('Wallet Window', layout)
def window_game_store():
    genres = ['Action', 'RPG', 'CityBuild', 'RogueLike', 'Simulation', 'Strategy']
    platforms = ['PC', 'PS', 'Xbox', 'Switch']
    listed_games = []
    below_prices = [6, 11, 16, 21, 26, 40, 50, 100]
    for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus, Developers.username FROM Games, Develop, Developers WHERE Games.GameID=Develop.GameID AND Develop.DeveloperID=Developers.DeveloperID AND ApprovalStatus=1 ORDER BY Genre'):
        listed_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[9]))
    layout = [[sg.Text('Select Genre', size=(10,1)), sg.Combo(genres, key='genre_selection')],
              [sg.Text('Below Price', size=(10,1)), sg.Combo(below_prices, key='below_price')],
              [sg.Text('Platforms', size=(10,1)), sg.Combo(platforms, key='platform_selection')],
              [sg.Text('Games:', size=(10,1)), sg.Listbox(listed_games, size=(30, 6), key='list_game'), sg.Button('Confirm')],
              [sg.Button('View Details'), sg.Button('Buy')],
              [sg.Button('Return to Main')]]
    return sg.Window('Game Store Window', layout)
              
def window_library():
    games = []
    stars = [1, 2, 3, 4, 5]
    for row in cur.execute('SELECT GameName, Games.GameID, Genre, PublishedDate, Price, Platforms, DownloadSize FROM Games, LibraryWithHas WHERE LibraryWithHas.GameID=Games.GameID AND PlayerID=?', (login_id,)):
        games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
    layout = [[sg.Text('Games you have', size=(20,1))],
              [sg.Listbox(games, size=(50,20), key = 'choosen_game')],
              [sg.Button('Gifted'), sg.Button('Bought')],
              [sg.Text('Rate', size=(10,1)), sg.Combo(stars, size=(30,1), key='choosen_star')],
              [sg.Button('Rate the Game')],
              [sg.Button('Return to Main')]]
    return sg.Window('Library Window', layout)

def window_create_game():
    genres_cre_game = ['Action', 'RPG', 'CityBuild', 'RogueLike', 'Simulation', 'Strategy', 'Action,RPG', 'Action,RogueLike', 'Simulation,Strategy']
    platforms_cre_game = ['PC', 'PS', 'Xbox', 'Switch', 'PC,PS', 'PC,Xbox', 'PC,Switch', 'PS,Xbox', 'PS,Switch', 'Xbox,Switch', 'PC,PS,Xbox', 'PC,PS,Xbox,Switch']
    layout = [[sg.Text('Game Name', size=(10,1)), sg.Input(size=(10,1), key='created_game_name')],
              # [sg.Text('Published Date', size=(10,1)), sg.Input(size=(10,1), key='cre_publisheddate_game')],
              [sg.Text('Published Date', size=(15,1)), sg.Input(key='cre_publisheddate_game', size=(15,1)), sg.CalendarButton('Choose Date', format='%Y-%m-%d')],
              [sg.Text('Price', size=(10,1)), sg.Input(size=(10,1), key='cre_price_game')],
              [sg.Text('Genre', size=(10,1)), sg.Combo(genres_cre_game, size=(15,1), key='cre_genre_game')],
              [sg.Text('Download Size', size=(10,1)), sg.Input(size=(10,1), key='cre_size_game')],
              [sg.Text('Compatible Platforms', size=(10,1)), sg.Combo(platforms_cre_game, key='cre_platforms_game')],
              [sg.Button('Create New Game')],
              [sg.Button('Return to Main')]]
    return sg.Window('Create Game Window', layout)

def window_send_gift():
    genres = ['Action', 'RPG', 'CityBuild', 'RogueLike', 'Simulation', 'Strategy']
    platforms = ['PC', 'PS', 'Xbox', 'Switch']
    below_prices_gift = [6, 11, 16, 21, 26, 40, 50, 100]
    friends = []
    giftable_games = []
    for row in cur.execute('select PlayerID, username from Players where PlayerID!=?', (login_id,)):
        friends.append((row[0], row[1])) 
    layout = [[sg.Text('Send a Gift to:', size=(15,1))],
              [sg.Listbox(friends, size=(25,15), key='choosen_friend'), sg.Button('Giftable Games')],
              [sg.Listbox(giftable_games, size=(45,15), key='choosen_gift')],
              [sg.Text('Genre', size=(10,1)), sg.Combo(genres, key='genre_selection_gift')],
              [sg.Text('Platforms', size=(10,1)), sg.Combo(platforms, key='platform_selection_gift')],
              [sg.Text('Below Price', size=(10,1)), sg.Combo(below_prices_gift, key='below_price_gift')],
              [sg.Button('List Games'), sg.Button('Gift')],
              [sg.Button('Return to Main')]]
    return sg.Window('Sending Gift Window', layout)

def window_gift():
    gifted_games = []
    for row in cur.execute('select Games.GameID, GameName from Games, GiftWithGifts where Games.GameID=GiftWithGifts.GameID AND ReceiverID=? AND GiftStatus=0', (login_id,)):
        gifted_games.append((row[0], row[1]))
    layout = [[sg.Text('Gifted Games', size=(10,1))],
              [sg.Listbox(gifted_games, size=(40,20), key='gifted_game')],
              [sg.Button('Accept'), sg.Button('Reject')],
              [sg.Button('Return to Main')]]
    return sg.Window('Gifted Games Window', layout)

def button_gifted(values):
    games = []
    for row in cur.execute('SELECT GameName, Games.GameID, Genre, PublishedDate, Price, Platforms, DownloadSize FROM Games, LibraryWithHas WHERE LibraryWithHas.GameID=Games.GameID AND PlayerID=? AND LibraryWithHas.ISgifted=1', (login_id,)):
        games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
    window.Element('choosen_game').Update(values=games)

def button_bought(values):
    games = []
    for row in cur.execute('SELECT GameName, Games.GameID, Genre, PublishedDate, Price, Platforms, DownloadSize FROM Games, LibraryWithHas WHERE LibraryWithHas.GameID=Games.GameID AND PlayerID=? AND LibraryWithHas.ISgifted=False', (login_id,)):
        games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
    window.Element('choosen_game').Update(values=games)

def button_accept(values):
    if not values['gifted_game']:
        sg.popup('First Select a game to accept.')
    else:
        game_id_gifted = values['gifted_game'][0][0]
        game_name_gifted = values['gifted_game'][0][1]
        cur.execute('select GameID from LibraryWithHas where PlayerID=? AND GameID=?', (login_id, game_id_gifted))
        row = cur.fetchone()
        if row is None:
            cur.execute('SELECT MAX(LibraryRecord) FROM LibraryWithHas')
            row = cur.fetchone()
            new_library_record = row[0] + 1
            cur.execute('INSERT INTO LibraryWithHas VALUES (?,?,?,1,0)', (new_library_record, game_id_gifted, login_id,))
            cur.execute('UPDATE GiftWithGifts SET GiftStatus=1 where ReceiverID=? AND GameID=?', (login_id, game_id_gifted))
            sg.popup('The game named: ' + game_name_gifted + ' has added to your library with library record: ' + str(new_library_record))
            gifted_games = []
            for row in cur.execute('select Games.GameID, GameName from Games, GiftWithGifts where Games.GameID=GiftWithGifts.GameID AND ReceiverID=? AND GiftStatus=0 AND Games.GameID!=?', (login_id, game_id_gifted)):
                gifted_games.append((row[0], row[1]))
            window.Element('gifted_game').Update(values=gifted_games)
        else:
            sg.popup('You already have this game in your library')

def button_reject(values):
    if not values['gifted_game']:
        sg.popup('First select a game to reject.')
    else:
        game_id_gifted = values['gifted_game'][0][0]
        gifted_games = []
        cur.execute('select Price from Games where GameID=?', (game_id_gifted,))
        gift_price = cur.fetchone()[0]
        cur.execute('select SenderID from GiftWithGifts where ReceiverID=? AND GameID=?', (login_id, game_id_gifted))
        gift_sender_id = cur.fetchone()[0]
        cur.execute('UPDATE Players SET PlayerWallet = PlayerWallet + ? WHERE PlayerID=?', (gift_price, gift_sender_id))
        sg.popup('You have rejected the gift and the money has refunded to sender.')
        cur.execute("UPDATE GiftWithGifts SET GiftStatus = 'abc' WHERE ReceiverID=? AND GameID=?", (login_id, game_id_gifted))
        for row in cur.execute('select Games.GameID, GameName from Games, GiftWithGifts where Games.GameID=GiftWithGifts.GameID AND ReceiverID=? AND GiftStatus=0 AND Games.GameID!=?', (login_id, game_id_gifted)):
            gifted_games.append((row[0], row[1]))
        window.Element('gifted_game').Update(values=gifted_games)


def button_list_games(values):
    friend_id = values['choosen_friend'][0][0]
    genre = values['genre_selection_gift']
    genre_select = '%' + genre + '%'
    below_this_price = values['below_price_gift']
    platform = values['platform_selection_gift']
    platform_select = '%' + platform + '%'
    giftable_games = []
    if genre_select is None:
        genre_select = '%Action,RPG,CityBuild,RogueLike,Simulation,Strategy%'
    if below_this_price is None:
        below_this_price = 100
    if platform is None:
        platform_select = '%PC,Xbox,PS,Switch%'
    for row in cur.execute('select Games.GameID, GameName, Genre, Price, PublishedDate, ApprovalStatus from Games where Games.ApprovalStatus=1 AND Genre LIKE ? AND Platforms LIKE ? AND Price < ? AND Games.GameID NOT IN (select LibraryWithHas.GameID from LibraryWithHas, Games where Games.GameID=LibraryWithHas.GameID AND Games.ApprovalStatus=1 AND LibraryWithHas.PlayerID=?)', (genre_select, platform_select, below_this_price, friend_id,)):
        giftable_games.append((row[0], row[1], row[2], row[3], row[4], row[5]))
    window.Element('choosen_gift').Update(values=giftable_games)
def button_giftable_games(values):
    giftable_games = []
    friend_id = values['choosen_friend'][0][0]
    friend_has = []
    for row in cur.execute('select GameID from LibraryWithHas where PlayerID=?', (friend_id,)):
        friend_has.append((row[0]))
    for row in cur.execute('select Games.GameID, GameName, Genre, Price, PublishedDate, ApprovalStatus from Games where Games.ApprovalStatus=1 AND Games.GameID NOT IN (select LibraryWithHas.GameID from LibraryWithHas, Games where Games.GameID=LibraryWithHas.GameID AND Games.ApprovalStatus=1 AND LibraryWithHas.PlayerID=?)', (friend_id,)):
        giftable_games.append((row[0], row[1], row[2], row[3], row[4], row[5]))
    window.Element('choosen_gift').Update(values=giftable_games)
              

def button_gift(values):
    if not values['choosen_gift']:
        sg.popup('Select a game to gift it.')
    else:
        gift_game = values['choosen_gift'][0]
        gift_game_id = gift_game[0]
        gift_friend_id = values['choosen_friend'][0][0]
        gift_game_name = gift_game[1]
        if gift_friend_id is None:
            sg.popup('Choose a friend to gift.')
        elif gift_game_id is None:
            sg.popup('Choose a game to gift.')
        else:
            cur.execute('select Price from Games where GameID=?', (gift_game_id,))
            row = cur.fetchone()
            if row is None:
                sg.popup('There is no game like this.')
            else:
                gift_game_price = row[0]
                cur.execute('select PlayerWallet from Players where PlayerID=?', (login_id,))
                row_player_wallet = cur.fetchone()[0]
                if row_player_wallet > gift_game_price:
                    cur.execute('UPDATE Players SET PlayerWallet = PlayerWallet - ? where PlayerID=?', (gift_game_price, login_id))
                    cur.execute('SELECT MAX(GiftID) FROM GiftWithGifts')
                    row_gift = cur.fetchone()
                    new_gift_id = row_gift[0] + 1
                    cur.execute('INSERT INTO GiftWithGifts VALUES (?,?,?,?,0)', (new_gift_id, login_id, gift_friend_id, gift_game_id))
                    sg.popup('This Game: ' + gift_game_name + ' succesfully sended to: ' + str(gift_friend_id) + ' Waiting for response.')
                else:
                    sg.popup('You do not have enough money to gift this game.')

    
    

def button_login(values):
    global login_id
    global login_user_name
    global login_user_type
    global window

    uname =values['username']
    upass = values['password']
    if uname == '':
        sg.popup('username cannot be empty')
    elif upass == '':
        sg.popup('Password cannot be empty')
    else:
        cur.execute('SELECT UserID, username FROM User WHERE username = ? AND password = ?', (uname, upass))
        row = cur.fetchone()
        if row is None:
            sg.popup('username or password is wrong. Try again.')
        else:
            login_id = row[0]
            login_user_name=row[1]
            cur.execute('SELECT PlayerID FROM Players WHERE PlayerID = ?', (login_id,))
            row_player = cur.fetchone()
            if row_player is None:
                cur.execute('SELECT DeveloperID FROM Developers WHERE DeveloperID = ?', (login_id,))
                row_developer = cur.fetchone()
                if row_developer is None:
                    cur.execute('select AdminID from Admins where AdminID = ?', (login_id,))
                    row_admin = cur.fetchone()
                    if row_admin is None:
                        sg.popup('No user found.')
                    else:
                        login_user_type = 'Admin'
                        sg.popup('Welcome ' + login_user_name + ' (Admin)')
                        window.close()
                        window = window_admin()
                else:
                    login_user_type='Developer'
                    sg.popup('Welcome ' + login_user_name + ' (Developer)')
                    window.close()
                    window = window_developer()
            else:
                login_user_type = 'Player'
                sg.popup('Welcome ' + login_user_name + ' (Player)')
                window.close()
                window=window_player()

def button_add_player(values):
    username = values['name']
    password = values['cpassword']
    gamertag = values['gamertag']
    if username == '':
        sg.popup('Username cannot be empty.')
    elif password == '':
        sg.popup('Password cannot be empty.')
    elif gamertag == '':
        sg.popup('Gamertag cannot be empty.')
    else:
        cur.execute('SELECT MAX(UserID) FROM User')
        row = cur.fetchone()
        new_id = row[0] + 1
        
        cur.execute('INSERT INTO User VALUES (?,?,?)', (new_id, username, password))
        
        cur.execute('INSERT INTO Players VALUES (?,?,?,0,?,null)', (new_id, username, password, gamertag))
        sg.popup('New player inserted username: ' + username + ' id: ' + str(new_id) + ' password: ' + password + ' Gamertag: ' + gamertag)
        window.Element('name').Update(value='')
        window.Element('cpassword').Update(value='')
        window.Element('gamertag').Update(value='')
def button_add(values):
    if not values['p_amount']:
        sg.popup('Enter an amount of money to add the wallet.')
    else:
        amount = int(values['p_amount'])
        if amount <= 0:
            sg.popup('enter positive amount.')
        else:
            cur.execute('UPDATE Players SET PlayerWallet = PlayerWallet + ? WHERE Players.PlayerID=?', (amount, login_id))
            sg.popup('This: ' + str(amount) + ' of money added to wallet.')
            window.Element('p_amount').Update(value='')
            cur.execute('SELECT PlayerWallet FROM Players WHERE PlayerID=?', (login_id,))
            row = cur.fetchone()
            login_credit = row[0]
            window.Element('login_credit').Update(value=str(login_credit))
def button_add_biography(values):
    bio_info = values['biography_text']
    if bio_info == '':
        sg.popup('Write something to add.')
    else:
        cur.execute('UPDATE Players SET Biography = ? WHERE Players.PlayerID=?', (bio_info, login_id))
        sg.popup('Biography updated.')
        cur.execute('SELECT Biography FROM Players WHERE PlayerID = ?', (login_id,))
        row = cur.fetchone()
        login_biography = row[0]
        window.Element('login_biography').Update(value=login_biography)
        
        
def button_rate_the_game(values):
    game = values['choosen_game']
    star = values['choosen_star']
    if game == '':
        sg.popup('Choose a game')
    elif star == '':
        sg.popup('Choose star out of 5 star to rate.')
    else:
        gameid = game[0][1]
        cur.execute('UPDATE LibraryWithHas SET Rating = ? WHERE GameID = ? AND PlayerID=?', (star, gameid, login_id))
        sg.popup('You have succesfully rated the game.')
        

def button_confirm(values):
    genre = values['genre_selection']
    genre_select = '%' + genre + '%'
    below_this_price = values['below_price']
    platform = values['platform_selection']
    platform_select = '%' + platform + '%'
    listed_games = []
    if values['genre_selection'] is None:
        genre_select = '%Action,RPG,CityBuild,RogueLike,Simulation,Strategy%'
    if values['below_price'] is None:
        below_this_price = 100
    if values['platform_selection'] is None:
        platform_select = '%PC,Xbox,PS,Switch%'
    for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus, Developers.username FROM Games, Develop, Developers WHERE Games.GameID=Develop.GameID AND Develop.DeveloperID=Developers.DeveloperID AND Genre LIKE ? AND Price < ? AND Games.Platforms LIKE ? ORDER BY Genre', (genre_select, below_this_price, platform_select)):
        listed_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[9]))
    window.Element('list_game').Update(values=listed_games)


def button_filter(values):
    genre_d = values['genre_selection_dev']
    genre_select_d = '%' + genre_d + '%'
    below_this_price_dev = values['below_price_d']
    platform_dev = values['platform_selection_dev']
    platform_selection = '%' + platform_dev + '%'
    status_appr = values['status_of_approve']
    listed_games_dev = []
    if not below_this_price_dev:
        below_this_price_dev = 200
    if not genre_d:
        genre_select_d = '%Action,RPG,CityBuild,RogueLike,Simulation,Strategy%'
    else:
        genre_select_d = '%' + genre_d + '%'
    if platform_dev:
        platform_selection = '%PC,Xbox,PS,Switch%'
    else:
        platform_selection = '%' + platform_dev + '%'
    if status_appr == 1:
        for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus FROM Games, Develop, Developers WHERE Developers.DeveloperID=Develop.DeveloperID AND Develop.GameID=Games.GameID AND  Developers.DeveloperID=? AND  Genre LIKE ? AND Price < ? AND Platforms LIKE ? AND ApprovalStatus = 1', (login_id, genre_select_d, below_this_price_dev, platform_selection)):
           listed_games_dev.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8]))
    elif status_appr == 0:
        for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus FROM Games, Develop, Developers WHERE Developers.DeveloperID=Develop.DeveloperID AND Develop.GameID=Games.GameID AND  Developers.DeveloperID=? AND  Genre LIKE ? AND Price < ? AND Platforms LIKE ? AND ApprovalStatus = 0', (login_id, genre_select_d, below_this_price_dev, platform_selection)):
            listed_games_dev.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8]))
    else:
        for row in cur.execute('SELECT Games.GameID, GameName, Genre, Price, PublishedDate, Platforms, DownloadSize, Sold, ApprovalStatus FROM Games, Develop, Developers WHERE Developers.DeveloperID=Develop.DeveloperID AND Develop.GameID=Games.GameID AND  Developers.DeveloperID=? AND  Genre LIKE ? AND Price < ? AND Platforms LIKE ?', (login_id, genre_select_d, below_this_price_dev, platform_selection)):
            listed_games_dev.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8]))
        
    window.Element('created_game').Update(values=listed_games_dev)





def button_list(values):
    genre_ad = values['genre_selection_admin']
    platform_ad = values['platform_selection_admin']
    price_ad = values['below_price_appr']
    size_ad = values['below_size_appr']
    before_date_ad = values['before_date']
    genre_select_ad = '%' + genre_ad + '%'
    platform_selection_ad = '%' + platform_ad + '%'
    approve_waiting_games = []
    if not genre_select_ad:
        genre_select_ad = '%Action,RPG,CityBuild,RogueLike,Simulation,Strategy%'
    if not platform_selection_ad:
        platform_selection_ad = '%PC,Xbox,PS,Switch%'
    if not price_ad:
        price_ad = 1000
    if not size_ad:
        size_ad = 1000000000
    if not before_date_ad:
        before_date_ad = '2500-01-01'
    for row in cur.execute("select GameID, GameName, Genre, Price, Platforms, DownloadSize, PublishedDate from Games WHERE Games.ApprovalStatus=0 AND Genre LIKE ? AND Platforms LIKE ? AND Price < ? AND DownloadSize < ? AND PublishedDate < ? ORDER BY PublishedDate DESC", (genre_select_ad, platform_selection_ad, price_ad, size_ad, before_date_ad)):
        approve_waiting_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
    window.Element('waiting_games').Update(values=approve_waiting_games)

    

def button_buy(values):
    if not values['list_game']:
        sg.popup('Select a game to buy.')
    else:
        game_buy = values['list_game'][0]
        game_id = game_buy[0]
        game_name = game_buy[1]
        game_cost = game_buy[3]
        cur.execute('SELECT LibraryRecord FROM LibraryWithHas WHERE LibraryWithHas.GameID=? AND LibraryWithHas.PlayerID=?', (game_id, login_id))
        row_libray = cur.fetchone()
        if row_libray is None:
            cur.execute('select PlayerWallet from Players where PlayerID=?', (login_id,))
            row_wallet = cur.fetchone()[0]
            if row_wallet > game_cost:
                cur.execute('UPDATE Players SET PlayerWallet = PlayerWallet - ? WHERE Players.PlayerID=?', (game_cost, login_id))
                cur.execute('UPDATE Developers SET DeveloperWallet = DeveloperWallet + ? WHERE DeveloperID = (SELECT DeveloperID FROM Develop WHERE GameID=?)', (game_cost, game_id))
                sg.popup('You have bought the game named ' + game_name + ' with the price of ' + str(game_cost) + ' dollars.')
                cur.execute('SELECT MAX(LibraryRecord) FROM LibraryWithHas')
                row = cur.fetchone()
                new_id = row[0] + 1
                cur.execute('INSERT INTO LibraryWithHas VALUES (?,?,?,0,0)', (new_id, game_id, login_id))
            else:
                sg.popup('You do not have enough money to buy this game.')
        else:
            sg.popup('You already have this game on Library.')


def button_view_details(values):
    if not values['list_game']:
        sg.popup('Select a game to view.')
    else:
        game_view = values['list_game'][0]
        game_id = game_view[0]
        cur.execute('select AVG(Rating) from LibraryWithHas where GameID=? AND Rating > 0' , (game_id,))
        row_rating = cur.fetchone()
        avg_rating = row_rating[0]
        if not game_view:
            sg.popup('Select a game to view')
        else:
            game_name = game_view[1]
            game_price = game_view[3]
            game_compatible = game_view[5]
            game_dev_name = game_view[9]
            game_size = game_view[6]
            sg.popup(' The Game named ' + game_name + ' has price: ' + str(game_price) + ' platforms: ' + game_compatible + ' developer name: ' + game_dev_name + ' game size: ' + str(game_size) + ' average rating: ' + str(avg_rating))
        
        
def button_add_to_balance(values):
    if not values['d_amount']:
        sg.popup('First enter an amount to add it to balance.')
    else:
        amount = int(values['d_amount'])
        if amount <= 0:
            sg.popup('enter positive amount.')
        else:
            cur.execute('UPDATE Developers SET DeveloperWallet = DeveloperWallet + ? WHERE Developers.DeveloperID=?', (amount, login_id))
            sg.popup('This: ' + str(amount) + ' of money added to balance.')
            window.Element('d_amount').Update(value='')
            cur.execute('SELECT DeveloperWallet FROM Developers WHERE DeveloperID=?', (login_id,))
            row = cur.fetchone()
            login_balance = row[0]
            window.Element('login_balance').Update(value=str(login_balance))

def button_create_new_game(values):
    # if values['created_game_name'] or values['cre_publisheddate_game'] or values['cre_price_game'] or values['cre_genre_game'] or values['cre_size_game'] or values['cre_platforms_game'] is None:
    #     sg.popup('Some informations about the game is lacking.')
    if values['cre_price_game'] == '':
        sg.popup('Some informations about the game is lacking.')
    elif values['cre_size_game'] == '':
        sg.popup('Some informations about the game is lacking.')
    elif values['created_game_name'] == '':
        sg.popup('Some informations about the game is lacking.')
    elif values['cre_publisheddate_game'] == '':
        sg.popup('Some informations about the game is lacking.')
    elif values['cre_genre_game'] == '':
        sg.popup('Some informations about the game is lacking.')
    elif values['cre_platforms_game'] == '':
        sg.popup('Some informations about the game is lacking.')
    else:
        approve_waiting_games = []
        a_game_name = values['created_game_name']
        a_game_date = values['cre_publisheddate_game']
        a_game_price = float(values['cre_price_game'])
        a_game_genre = values['cre_genre_game']
        a_game_size = int(values['cre_size_game'])
        a_game_platform = values['cre_platforms_game']
        cur.execute('select MAX(GameID) FROM Games')
        row=cur.fetchone()
        new_game_id = row[0] + 1
        cur.execute('INSERT INTO Games VALUES (?,0,?,?,?,?,?,0,?)', (new_game_id, a_game_genre, a_game_date, a_game_price, a_game_platform, a_game_size, a_game_name))
        cur.execute('INSERT INTO Develop VALUES (?, ?)', (login_id, new_game_id))
        sg.popup('Succesfully created game named ' + a_game_name + '.')

def button_approve(values):
    if not values['waiting_games']:
        sg.popup('First select a game to approve it.')
    else:
        game_id = values['waiting_games'][0][0]
        approve_waiting_games = []
        cur.execute('UPDATE Games SET ApprovalStatus = 1 where GameID = ? ', (game_id,))
        for row in cur.execute("select GameID, GameName, Genre, Price, Platforms, DownloadSize, PublishedDate from Games WHERE Games.ApprovalStatus=0 ORDER BY PublishedDate"):
            approve_waiting_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
        window.Element('waiting_games').Update(values=approve_waiting_games)
        
        sg.popup('You have approved the game for listing.')


def button_revoke(values):
    if not values['waiting_games']:
        sg.popup('First select a game to revoke it.')
    else:
        game_id = values['waiting_games'][0][0]
        approve_waiting_games = []
        cur.execute("UPDATE Games SET ApprovalStatus = 'abc' where GameID = ? ", (game_id,))
        for row in cur.execute('select GameID, GameName, Genre, Price, Platforms, DownloadSize, PublishedDate from Games WHERE Games.ApprovalStatus=0 ORDER BY PublishedDate'):
            approve_waiting_games.append((row[0], row[1], row[2], row[3], row[4], row[5], row[6]))
        window.Element('waiting_games').Update(values=approve_waiting_games)
        sg.popup('The game you selected has revoked.')
    


window = window_main()
while True:
    event, values = window.read()
    if event == 'Login page':
        window.close()
        window = window_login()
    elif event == 'Login':
        window.close()
        button_login(values)
    elif event == 'Create new player':
        window.close()
        window = window_create_player()
    elif event == 'Created Games':
        window.close()
        window = window_created_games()
    elif event == 'View Balance':
        window.close()
        window = window_view_balance()
    elif event == 'My Account':
        window.close()
        window = window_my_account()
    elif event == 'View My Wallet':
        window.close()
        window = window_view_wallet()
    elif event == 'Gifted Games':
        window.close()
        window = window_gift()
    elif event == 'Gift':
        button_gift(values)
    elif event == 'Friends':
        window.close()
        window = window_send_gift()
    elif event == 'Games':
        window.close()
        window = window_game_store()
    elif event == 'Giftable Games':
        button_giftable_games(values)
    elif event == 'Accept':
        button_accept(values)
    elif event == 'Reject':
        button_reject(values)
    elif event == 'Library':
        window.close()
        window = window_library()
    elif event == 'Add Player':
        button_add_player(values)
    elif event == 'Add':
        button_add(values)
    elif event == 'Add Biography':
        button_add_biography(values)
    elif event == 'Add to Balance':
        button_add_to_balance(values)
    elif event == 'Rate the Game':
        button_rate_the_game(values)
    elif event == 'Confirm':
        button_confirm(values)
    elif event == 'Buy':
        button_buy(values)
    elif event == 'List':
        button_list(values)
    elif event == 'Create Game':
        window.close()
        window = window_create_game()
    elif event == 'Create New Game':
        button_create_new_game(values)
    elif event == 'List Games':
        button_list_games(values)
    elif event == 'Games Waiting Approval':
        window.close()
        window = window_approval()
    elif event == 'Return to Create':
        window.close()
        window = window_main()
    elif event == 'Filter':
        button_filter(values)
    elif event == 'Gifted':
        button_gifted(values)
    elif event == 'Bought':
        button_bought(values)
    elif event == 'View Details':
        button_view_details(values)
    elif event == 'Approve':
        button_approve(values)
    elif event == 'Revoke':
        button_revoke(values)
    elif event == 'Return to Main':
        if login_user_type == 'Player':
            window.close()
            window = window_player()
        elif login_user_type == 'Developer':
            window.close()
            window = window_developer()
        elif login_user_type == 'Admin':
            window.close()
            window = window_admin()
    elif event == 'Logout':
        login_id = -1
        login_user_name = -1
        login_user_type = -1
        window.close()
        window=window_login()
    elif event == sg.WIN_CLOSED:
        break
window.close()
con.commit()
con.close()
                
            
              
                                                    
                                                    